# **RocketPy Simulation**
Create one venv for each project, so each one can have it's own RocketPy version and thus the simulation results do not change if you run it again later.

Set `PROJECT` in **Configuration** to desired one before running the simulation.

In [ ]:
from importlib.metadata import version
print("RocketPy version:", version("rocketpy"))

## Imports

In [ ]:
import os
from pathlib import Path

# move one level up so simulation_core and projects are found.
os.chdir(next(folder for folder in [Path.cwd(), *Path.cwd().parents] if (folder / "simulation_core").is_dir()))

import simulation_core.simulation as sim
import simulation_core.outputs as outputs
import simulation_core.utils as utils
import simulation_core.reanalysis as reanalysis
from openmeteo_requests import OpenMeteoRequestsError

utils.show_errors_without_traceback(OpenMeteoRequestsError)

In [ ]:
# reload imported Python modules when you change their .py files
%load_ext autoreload
%autoreload 2

## Configuration
The length unit chosen here is millimeters to keep the values more readable. If necessary, values should be converted accordingly.

In [ ]:

PROJECT = "ALBATROSS_II"
CONFIG_PATH = Path(f"projects/{PROJECT}/config.toml")
ZONES_PATH = Path(f"projects/{PROJECT}/zones.toml")

# True  -> use ONLY the standard atmosphere
# False -> use every envType from config EXCEPT the standard atmosphere (Windy / custom / etc.)
USE_ONLY_STANDARD_ENVIRONMENT = False

params = utils.load_config(CONFIG_PATH)

env_types = utils.ensure_list(params.config.environment.envType)
if USE_ONLY_STANDARD_ENVIRONMENT:
    env_types = ["standard_atmosphere"]
else:
    env_types = [e for e in env_types if e != "standard_atmosphere"]
params.config = params.config.model_copy(update={
    "environment": params.config.environment.model_copy(update={"envType": env_types})
})

print("Config loaded")
print("Active envTypes:", params.config.environment.envType)
print(f"Simulating for "
      f"{params.config.environment.date if type(params.config.environment.date) == str else params.config.environment.date.strftime('%d.%m.%Y, %H:%M:%S')}, "
      f"{params.config.environment.timezone}")

## Zones

In [ ]:
exclusion_zones, buffer_zones, exclusion_zone_safety_margin = utils.load_zones(ZONES_PATH)
# Add buffer zones around exclusion zones
buffer_zones.update(utils.scale_zones(exclusion_zones, exclusion_zone_safety_margin))

# buffer_zones = {}                # remove all buffer zones; check only hard exclusions
# buffer_zones = exclusion_zones   # use exclusions as buffers (no margin, discard buffer zones from zones.toml)

outputs.plot_landing_positions_with_modes(params, exclusion_zones, buffer_zones, plot_name="Zones", zones_only=True)

## Environments Initialization
In this section the environments are initialized.

In [ ]:
sim.create_environment(params)

## Simulation
### Tanks / Engine

In [ ]:
sim.create_engine(params)

### Rocket
Create Nosecone, Tailcone, Fins, Parachutes.

RocketPy definitions:
- dry mass = rocket with motor but without propellant
- Rocket Loaded Mass = Wet mass
- Rocket Center of Dry Mass - Nozzle Exit = Rocket Center of Dry Mass from bottom

In [ ]:
sim.create_rocket(params)

## Flight

In [ ]:
sim.create_flight(params)

## Output

A heading direction is only safe if <u>every</u> simulated rocket scenario (nominal/no main/ballistic/payload), every inclination and every environment stays outside the buffer zones.

In [ ]:
reanalysis.build_reanalysis_artifacts(params)

outputs.run_notebook_display_mode(params, exclusion_zones, buffer_zones)

reanalysis.run_reanalysis_comparison(params)

if params.export_kml:
    exported_kml_files = outputs.export_all_kml(params)
    # exported_csv_files = outputs.export_all_trajectory_csv(params)

In [ ]:
# needs its own cell because it waits for the notebook to be saved by the IDE before exporting to HTML
outputs.export_notebook_to_html("jupyternb/simulation_orchestration.ipynb", params.project_path / "report.html")